In [1]:
# CONFIG — autores do Domínio Público BR (data/dpbr_*/, baixados pelo builder)
import glob, os, re
from pathlib import Path

RAW_DIRS = sorted(glob.glob('data/dpbr_*'))     # um dir por autor (data/dpbr_<autor>)
TOKENIZER = "artifacts/tokenizers/gpt2_ptbr_50k_v2"
WEIGHT_CLIP = 500_000
SPLIT_FRAC = 0.85
RANDOM_STATE = 1
MIN_TEXT = 3_000          # abaixo disso = scan/vazio -> descarta
print('dirs:', RAW_DIRS)
for d in RAW_DIRS:
    print(' ', d, len(list(Path(d).glob('*.pdf'))), 'pdfs')

dirs: ['data\\dpbr_alencar', 'data\\dpbr_artur_azevedo', 'data\\dpbr_caminha', 'data\\dpbr_julia_lopes', 'data\\dpbr_lima_barreto', 'data\\dpbr_macedo', 'data\\dpbr_qorpo_santo', 'data\\dpbr_taunay', 'data\\dpbr_voltaire']
  data\dpbr_alencar 27 pdfs
  data\dpbr_artur_azevedo 123 pdfs
  data\dpbr_caminha 4 pdfs
  data\dpbr_julia_lopes 6 pdfs
  data\dpbr_lima_barreto 33 pdfs
  data\dpbr_macedo 15 pdfs
  data\dpbr_qorpo_santo 7 pdfs
  data\dpbr_taunay 10 pdfs
  data\dpbr_voltaire 11 pdfs


In [2]:
# versionamento automático: base ATUAL = última vN detectada; saída = vN+1
vers = sorted(int(re.search(r'df_full_v(\d+)\.pq', p).group(1))
              for p in glob.glob('data/df_full_v*.pq'))
BASE_TXT = f'data/df_full_v{max(vers)}.pq'
BASE_ENC = f'data/df_full_encoded_v{max(vers)}.pq'
NEXT_VERSION = max(vers) + 1
OUT_TXT = f'data/df_full_v{NEXT_VERSION}.pq'
OUT_ENC = f'data/df_full_encoded_v{NEXT_VERSION}.pq'
print(f'Bases: v{vers} | base ATUAL: {BASE_TXT} | nova: v{NEXT_VERSION}')

Bases: v[0, 21, 22, 23, 24] | base ATUAL: data/df_full_v24.pq | nova: v25


In [3]:
import pdfplumber

def pdf_to_txt(pdf_path, txt_path):
    with pdfplumber.open(pdf_path) as pdf:
        text = "".join((pg.extract_text() or "") + "\n" for pg in pdf.pages)
    text = re.sub(r"\n{2,}", "\n\n", text)
    text = re.sub(r" +", " ", text).strip()
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(text)
    return len(text)

meta_pdf = []
for rd in RAW_DIRS:
    autor = Path(rd).name.replace('dpbr_', '')
    txt_dir = Path(f'data/{autor}_txt')
    txt_dir.mkdir(exist_ok=True)
    for f in sorted(Path(rd).glob('*.pdf')):
        # título limpo do nome do arquivo: <co_obra>_<slug>.pdf
        title = Path(f).stem
        m = re.match(r'\d+_(.*)$', title)
        title = m.group(1) if m else title
        txt = txt_dir / f'{Path(f).stem}.txt'
        try:
            n = pdf_to_txt(f, txt)
            meta_pdf.append({'title': title, 'author': autor, 'path_raw': str(f),
                             'path_txt': str(txt), 'text_raw_len': n})
        except Exception as e:
            print('  ERRO', f, e)
print('pdfs extraídos:', len(meta_pdf))

pdfs extraídos: 236


In [4]:
import pandas as pd
df_new = pd.DataFrame(meta_pdf)
df_new['class'] = df_new['author']
df_new['extension'] = 'pdf'
df_new['subtitle'] = ""
df_new['text'] = df_new.path_txt.apply(lambda x: Path(x).read_text(encoding='utf-8'))
df_new['text_len'] = df_new['text'].str.len()
n0 = (df_new['text_len'] == 0).sum()
if n0:
    print(f'AVISO: {n0} docs sem texto (scan puro). Descartando:')
    for t in df_new.loc[df_new.text_len == 0, 'title']:
        print('  -', t)
    df_new = df_new[df_new.text_len > 0].copy()
nscan = (df_new['text_len'] < MIN_TEXT).sum()
if nscan:
    print(f'AVISO: {nscan} docs < {MIN_TEXT} chars (scan/vazio). Descartando:')
    for t in df_new.loc[df_new.text_len < MIN_TEXT, 'title']:
        print('  -', t)
    df_new = df_new[df_new.text_len >= MIN_TEXT].copy()
df_new = df_new.sort_values('text_raw_len', ascending=False).reset_index(drop=True)
print(df_new[['author','title','text_raw_len']])

AVISO: 5 docs sem texto (scan puro). Descartando:
  - a_isca
  - amelia_smith
  - dias_de_guerra_e_de_sertao
  - paisagens_brasileiras
  - reminiscencias
AVISO: 24 docs < 3000 chars (scan/vazio). Descartando:
  - a_conselho_do_marido
  - a_doenca_do_fabricio
  - pequetita
  - as_escuras
  - as_paradas
  - barca
  - duas_apostas
  - elefantes_e_ursos
  - em_sonhos
  - mal_por_mal
  - na_exposicao
  - o_cuco
  - o_galo
  - o_lencinho
  - o_paulo
  - o_retrato
  - o_ultimo_palpite
  - octogenario
  - os_compadres
  - paga_ou_morre
  - pan_americano
  - piedade_filial
  - puelina
  - um_don_juan_de_provincia
            author                  title  text_raw_len
0           macedo           o_moco_loiro        665517
1          alencar              o_guarani        622215
2           macedo         os_dois_amores        591962
3           macedo     as_vitimas_algozes        577893
4          alencar            o_sertanejo        543849
..             ...                    ...           

In [5]:
from src.prep import clean_text2
df_new['text_clean'] = df_new.text.apply(clean_text2)
df_new['text_clean_len'] = df_new['text_clean'].str.len()
print(df_new[['author','title','text_raw_len','text_clean_len']])

            author                  title  text_raw_len  text_clean_len
0           macedo           o_moco_loiro        665517          665517
1          alencar              o_guarani        622215          622214
2           macedo         os_dois_amores        591962          591962
3           macedo     as_vitimas_algozes        577893          577893
4          alencar            o_sertanejo        543849          544141
..             ...                    ...           ...             ...
202  artur_azevedo      uma_carga_de_sono          3272            3272
203  artur_azevedo  denuncia_involuntaria          3185            3185
204  artur_azevedo  historia_de_um_domino          3060            3060
205  artur_azevedo  assunto_para_um_conto          3017            3017
206   lima_barreto   a_mulher_de_anacleto          3008            3006

[207 rows x 4 columns]


In [6]:
# Diagnóstico BR x PT-PT x EN + arcaísmo. Docs com marcadores EN fortes saem da base.
MARKS_BR = ['você', 'fato,', 'direção', 'celular', 'a gente', 'não sei']
MARKS_PT = ['facto,', 'direcção', 'telemóvel', 'connosco', 'comboio', 'tu ']
MARKS_EN = [' the ', ' and ', ' of the ', ' was ', 'chapter ']
ARCAIC   = ['ph', 'th', 'annos', 'diccionario', 'escriptor', 'secção', 'licção', 'dous', 'typo', 'sciencia']
def diag(t):
    tl = ' ' + t.lower().replace('\n', ' ') + ' '
    br = sum(tl.count(m) for m in MARKS_BR)
    pt = sum(tl.count(m) for m in MARKS_PT)
    en = sum(tl.count(m) for m in MARKS_EN)
    arch = sum(tl.count(m) for m in ARCAIC) / max(len(tl), 1) * 1000
    var = 'EN' if en > max(br, pt) * 10 else ('BR' if br >= pt else ('PT-PT?' if pt > 0 else '?'))
    return var, round(arch, 2)
df_new[['variante','arch_per_mil']] = df_new.text_clean.apply(lambda t: pd.Series(diag(t)))
nen = (df_new.variante == 'EN').sum()
if nen:
    print(f'AVISO: {nen} docs em INGLÊS (catalogados como pt por engano). Descartando:')
    for t in df_new.loc[df_new.variante == 'EN', 'title']:
        print('  -', t)
    df_new = df_new[df_new.variante != 'EN'].copy()
print(df_new[['author','title','variante','arch_per_mil','text_clean_len']])
print()
print('docs por autor:', df_new.author.value_counts().to_dict())

            author                  title variante  arch_per_mil  \
0           macedo           o_moco_loiro   PT-PT?          0.00   
1          alencar              o_guarani   PT-PT?          0.00   
2           macedo         os_dois_amores   PT-PT?          0.00   
3           macedo     as_vitimas_algozes   PT-PT?          0.00   
4          alencar            o_sertanejo       BR          0.00   
..             ...                    ...      ...           ...   
202  artur_azevedo      uma_carga_de_sono       BR          0.00   
203  artur_azevedo  denuncia_involuntaria       BR          0.00   
204  artur_azevedo  historia_de_um_domino       BR          0.00   
205  artur_azevedo  assunto_para_um_conto       BR          0.00   
206   lima_barreto   a_mulher_de_anacleto       BR          0.66   

     text_clean_len  
0            665517  
1            622214  
2            591962  
3            577893  
4            544141  
..              ...  
202            3272  
203    

In [7]:
cols = ['title','author','extension','class','subtitle','path_raw','path_txt',
        'text_raw_len','text','text_len','text_clean','text_clean_len','weights','split']
df_base = pd.read_parquet(BASE_TXT)
print('base atual:', BASE_TXT, '| docs:', len(df_base), '| autores:', df_base.author.value_counts().to_dict())
assert set(cols) <= set(df_base.columns), 'schema inesperado da base'

# dedupe por título contra a base (evita obra que já entrou via Wikisource/outra fonte)
import unicodedata
def norm_t(t):
    t = unicodedata.normalize('NFD', str(t).lower())
    t = ''.join(c for c in t if not unicodedata.combining(c))
    return re.sub(r'[^a-z0-9 ]', ' ', t).strip()
base_titles = set(df_base.title.apply(norm_t))
dup = df_new[df_new.title.apply(norm_t).isin(base_titles)]
if len(dup):
    print(f'AVISO: {len(dup)} títulos já existem na base (dedupe). Pulando:')
    for t in dup.title:
        print('  -', t)
    df_new = df_new[~df_new.title.apply(norm_t).isin(base_titles)].copy()

df_new['weights'] = 0.0
df_new['split'] = ''
df_full = pd.concat([df_base[cols], df_new[cols]], ignore_index=True)
df_full['text_len'] = df_full['text'].str.len()
df_full['text_clean'] = df_full.text.apply(clean_text2)
df_full['text_clean_len'] = df_full['text_clean'].str.len()
print('total docs:', len(df_full), '| chars clean M:', round(df_full.text_clean_len.sum()/1e6, 2))

base atual: data/df_full_v24.pq | docs: 1145 | autores: {'machado': 349, 'campos': 171, 'cruz_sousa': 166, 'lovecraft': 114, 'bilac': 98, 'alvares': 90, 'king': 65, 'lobato': 20, 'augusto_anjos': 14, 'joao_rio': 11, 'guimaraes': 11, 'tolkien': 10, 'aluisio': 9, 'poe': 5, 'coelho_neto': 4, 'pompeia': 4, 'arinos': 2, 'chambers': 1, 'teofilo': 1}
AVISO: 1 títulos já existem na base (dedupe). Pulando:
  - uma_por_outra


total docs: 1351 | chars clean M: 100.06


In [8]:
df_full['weights'] = df_full['text_clean_len'].clip(0, WEIGHT_CLIP)
df_full['weights'] = df_full['weights'] / df_full['weights'].sum()
index_eval = df_full.sample(frac=1-SPLIT_FRAC, random_state=RANDOM_STATE, weights='weights').index
df_full['split'] = pd.Series(df_full.index.isin(index_eval)).map({False: 'train', True: 'eval'})
print(df_full.groupby(['author','split']).weights.sum())

author         split
alencar        eval     0.075759
               train    0.010495
aluisio        train    0.000264
alvares        eval     0.001240
               train    0.004214
arinos         train    0.000312
artur_azevedo  eval     0.020524
               train    0.026428
augusto_anjos  eval     0.000072
               train    0.000370
bilac          eval     0.000274
               train    0.002711
caminha        eval     0.014505
campos         eval     0.000221
               train    0.006794
chambers       eval     0.007278
coelho_neto    train    0.000051
cruz_sousa     eval     0.000372
               train    0.005713
guimaraes      eval     0.000052
               train    0.000258
joao_rio       train    0.001185
julia_lopes    eval     0.016307
               train    0.003071
king           eval     0.433432
               train    0.017812
lima_barreto   eval     0.058450
               train    0.010579
lobato         train    0.003256
lovecraft      eval   

In [9]:
df_full = df_full.sample(frac=1).reset_index(drop=True)  # shuffle
df_full.to_parquet(OUT_TXT)
import os
print('salvo:', OUT_TXT, round(os.path.getsize(OUT_TXT)/1e6, 1), 'MB')
print('base antiga intocada:', BASE_TXT)

salvo: data/df_full_v25.pq 130.0 MB
base antiga intocada: data/df_full_v24.pq


In [10]:
from transformers import AutoTokenizer
from tqdm import tqdm
tqdm.pandas()
tok = AutoTokenizer.from_pretrained(TOKENIZER)
print('tokenizer:', TOKENIZER, '| vocab:', tok.vocab_size)
encode = lambda s: tok(s, truncation=False).input_ids
df_full['text_encoded'] = df_full.text_clean.progress_apply(lambda x: encode(x))
df_full['text_encoded_len'] = df_full.text_encoded.apply(len)
nc = int(df_full['text_clean_len'].sum()); nt = int(df_full['text_encoded_len'].sum())
print(f'chars {nc/1e6:.2f}M -> tokens {nt/1e6:.2f}M | compressão chars/tok = {nc/nt:.2f}')
bal = df_full.groupby('author').agg(docs=('title','count'),
                                    chars_M=('text_clean_len', lambda s: round(s.sum()/1e6,2)),
                                    toks_M=('text_encoded_len', lambda s: round(s.sum()/1e6,2)),
                                    pct_chars=('text_clean_len', lambda s: round(100*s.sum()/nc,1)))
print(bal.sort_values('chars_M', ascending=False))

C:\Users\Bruno\.conda\envs\transformers-fun\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


tokenizer: artifacts/tokenizers/gpt2_ptbr_50k_v2 | vocab: 50257


  0%|          | 0/1351 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (5018 > 1024). Running this sequence through the model will result in indexing errors


  2%|▏         | 21/1351 [00:00<00:20, 66.40it/s]

...

 97%|█████████▋| 1311/1351 [00:36<00:01, 22.79it/s]

 99%|█████████▉| 1343/1351 [00:36<00:00, 39.71it/s]

100%|█████████▉| 1348/1351 [00:37<00:00, 34.88it/s]

100%|██████████| 1351/1351 [00:37<00:00, 36.31it/s]

chars 100.06M -> tokens 23.48M | compressão chars/tok = 4.26
               docs  chars_M  toks_M  pct_chars
author                                         
king             65    59.93   13.38       59.9
alencar          27     5.98    1.56        6.0
tolkien          10     4.96    1.09        5.0
lima_barreto     33     4.64    1.19        4.6
macedo           15     4.18    1.11        4.2
machado         349     3.89    1.00        3.9
lovecraft       114     3.29    0.66        3.3
artur_azevedo    98     3.15    0.93        3.2
poe               5     2.80    0.65        2.8
taunay            6     1.32    0.35        1.3
julia_lopes       5     1.30    0.33        1.3
voltaire         11     1.15    0.30        1.1
caminha           4     0.97    0.25        1.0
chambers          1     0.49    0.12        0.5
campos          171     0.47    0.12        0.5
cruz_sousa      166     0.41    0.11        0.4
alvares          90     0.37    0.11        0.4
lobato           20     0.2

In [11]:
df_full.to_parquet(OUT_ENC)
import os
print('salvo:', OUT_ENC, round(os.path.getsize(OUT_ENC)/1e6, 1), 'MB')
df_old = pd.read_parquet(BASE_ENC)
print()
print(f'== VALIDAÇÃO v{max(vers)} -> v{NEXT_VERSION} ==')
print('docs:', len(df_old), '->', len(df_full), f'(+{len(df_full)-len(df_old)})')
print('tokens M:', round(df_old.text_encoded_len.sum()/1e6, 2), '->', round(df_full.text_encoded_len.sum()/1e6, 2))
novos = [a for a in df_full.author.unique() if a not in set(df_old.author.unique())]
print('autores novos:', novos)
print()
print('PRÓXIMO PASSO: apontar o yaml para o v' + str(NEXT_VERSION))
print('  path_input_encoded:', OUT_ENC)

salvo: data/df_full_encoded_v25.pq 172.0 MB



== VALIDAÇÃO v24 -> v25 ==
docs: 1145 -> 1351 (+206)
tokens M: 17.42 -> 23.48
autores novos: ['artur_azevedo', 'macedo', 'lima_barreto', 'qorpo_santo', 'alencar', 'julia_lopes', 'voltaire', 'taunay', 'caminha']

PRÓXIMO PASSO: apontar o yaml para o v25
  path_input_encoded: data/df_full_encoded_v25.pq
